# Lun 12/10 · 08:00 — TP d'introduction : installation vérifiée et premier script

Lun 12/10, 08:50–09:30. L'installation se fait avant le cours, avec `tp/c01-installation.md`. En salle, ces 40 min servent à vérifier et à réparer.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt personnel `fleetcheck-<nom>`.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- Main levée dès qu'une vérification reste en `KO` : l'enseignant passe.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## 1. Python, JupyterLab et git — fin 09:00

Le notebook tourne avec le Python 3.14 de votre environnement virtuel `.venv`.

In [ ]:
print(sys.version)
check(sys.version_info[:2], (3, 14))
check(sys.prefix != sys.base_prefix, True)

La seconde vérification est `KO` si le noyau n'est pas celui du venv : relancez JupyterLab avec `.venv/bin/python -m jupyter lab` (Windows : `.venv\Scripts\python -m jupyter lab`).

Dans le terminal de JupyterLab, `git --version` affiche une version.

## 2. Dépôts — fin 09:15

Votre dépôt personnel est cloné dans `esgi-python`, le dépôt public `fleetcheck` à côté (`tp/c01-installation.md`, section 4).

In [ ]:
from pathlib import Path

check(Path("../fleetcheck/simserver.py").exists(), True)

## 3. Premier script — fin 09:25

Créez `hello.py` à la racine de votre dépôt (lanceur de JupyterLab, *Python File*, puis renommer) :

```text
import platform
import sys

print(platform.system())
print(sys.version)
```

`%run hello.py` l'exécute : le système (`Darwin`, `Linux` ou `Windows`), puis une version qui commence par `3.14`.

In [ ]:
%run hello.py

In [ ]:
r = subprocess.run([sys.executable, "hello.py"], capture_output=True, encoding="utf-8")
lines = r.stdout.splitlines()
check(len(lines), 2)
check(lines[0] in ("Darwin", "Linux", "Windows"), True)
check(lines[1].startswith("3.14"), True)

JupyterLab crée des fichiers de sauvegarde (`.ipynb_checkpoints/`) et Python des caches (`__pycache__/`) : ils ne vont pas dans git. Cette cellule écrit le `.gitignore` :

In [ ]:
Path(".gitignore").write_text(".ipynb_checkpoints/\n__pycache__/\n");

Enregistrez ce notebook, puis dans le terminal de JupyterLab, depuis votre dépôt :

```text
git add .gitignore hello.py c01.ipynb
git commit -m "Installation vérifiée"
git push
```

`git commit` affiche « Please tell me who you are » ou « configured automatically » : l'identité n'est pas configurée (section 2 du guide). Configurez-la, puis `git commit --amend --reset-author` et `git push`. `git push` refuse le mot de passe : jeton ou Git Credential Manager, section 4.

In [ ]:
!git log --oneline -1

L'auteur du commit doit être votre nom et l'adresse de votre compte GitHub ou GitLab, sinon le commit n'est pas relié à votre compte :

In [ ]:
!git log -1 --format='%an <%ae>'

La cellule affiche votre commit.

## 4. Bilan — fin 09:30

Votre `hello.py` et ce notebook sont poussés. Sinon, l'enseignant vous place à côté d'un voisin équipé pour la séance suivante (lun 12/10, 09:45), et votre installation se règle pendant son TP.

## Pour aller plus loin

Dans le terminal de JupyterLab, depuis `fleetcheck` : `python3.14 simserver.py` (Windows : `py -3.14 simserver.py`), puis ouvrir `http://127.0.0.1:8001/health` dans le navigateur : premier contact avec le serveur du chapitre 5. Ctrl+C l'arrête.